In [ ]:
#RAKE

In [2]:
import pymorphy2 as pm
import codecs
import string

In [3]:
m = pm.MorphAnalyzer()

In [1]:
words = []
words2 = ""

In [4]:
with open('IMS2024\MitrofanovaAdamova_IMS_2024_rus.txt', 'r', encoding='utf-8') as f: #прописываем путь к корпусу (папка или файл)
    text = f.read()
    words = text.split()
    for i in range(len(words) - 1):
        if m.parse(words[i])[0].tag.POS == "ADVB" or m.parse(words[i])[0].tag.POS == "VERB" or m.parse(words[i][:-1])[0].tag.POS == "VERB" or m.parse(words[i])[0].tag.POS == "PRTS"or m.parse(words[i])[0].tag.POS == "INFN"or m.parse(words[i])[0].tag.POS == "COMP" or m.parse(words[i])[0].tag.POS == "ADJS" or m.parse(words[i])[0].tag.POS == "GRND"or m.parse(words[i])[0].tag.POS == "CONJ":
            words2 += " | " + words[i] + " | "
        elif (m.parse(words[i])[0].tag.POS == "NOUN" and m.parse(words[i])[0].tag.case == ("gent" or "accs")):
            words2 += words[i] + " | "
        elif m.parse(words[i])[0].tag.POS == "NOUN" and m.parse(words[i + 1])[0].tag.POS == ("NOUN" or "ADJF") and m.parse(words[i])[0].tag.case != m.parse(words[i + 1])[0].tag.case and m.parse(words[i + 1])[0].tag.case != "gent":
            words2 += words[i] + " | "
        elif (m.parse(words[i])[0].tag.POS == "NOUN" and m.parse(words[i])[0].tag.case != ("nomn" or "accs") and m.parse(words[i + 1])[0].tag.POS == "NOUN" and m.parse(words[i + 1])[0].tag.case == ("nomn" or "accs")):
            words2 += words[i] + " | "
        elif (m.parse(words[i])[0].tag.POS == None and m.parse(words[i][:-1])[0].tag.POS == None and m.parse(words[i][1:])[0].tag.POS == None):
            words2 += " | " + words[i] + " | "
        elif (m.parse(words[i])[0].tag.POS == "NOUN" and m.parse(words[i - 1])[0].tag.POS == "ADJF" or m.parse(words[i - 1])[0].tag.POS == "PRTF") and m.parse(words[i])[0].tag.case == m.parse(words[i + 1])[0].tag.case:
            words2 += words[i] + " | "
        else:
            words2 += words[i] + " "

In [5]:
words2

' | 1. | Введение Проект, представленный в данной статье, |  | посвящен | памяти |  | основателя |  | и |  | руководителя | Петербургской школы | корпусной  | и | компьютерной лингвистики | Виктора | Павловича | Захарова, нашего учителя |  | и | коллеги, который с  | 2002 | года |  | был | главным организатором конференций |  | и | семинаров,  | где |  | обсуждались | проблемы создания |  | и |  | применения | корпусов | текстов. За двадцатилетний период проведения | научных встреч |  | были |  | собраны | ценные материалы, которые  | связаны | с историей корпусной  | и | компьютерной лингвистики, с развитием основных направлений, с  | кругом | проблем |  | и | предлагаемых решений, | с исследованием этапов |  | становления |  | и |  | изменений | терминологии рассматриваемой предметной области,  | ее | логико-понятийной схемы |  | и | принципов | стандартизации. Цель проекта |  | состояла | в разработке комплексного корпусного  | и | терминологического ресурса | с возможностью многопа

In [6]:
with open('Rake_Text_MA.txt', 'w', encoding='utf-8') as f2:
    f2.write(words2)

In [7]:
from __future__ import division
import operator
import nltk

In [16]:
punct = open("punctuation_marks.txt", 'r', encoding='utf-8').read()

In [17]:
punct_list = nltk.word_tokenize(punct)

In [18]:
stop = open("stopwords-ru.txt", 'r', encoding='utf-8').read()

In [19]:
stop_list = nltk.word_tokenize(stop) + nltk.word_tokenize(punct)

In [20]:
def isPunct(word):
  return len(word) == 1 and (word in string.punctuation or word in punct_list)

In [21]:
def isNumeric(word):
  try:
    float(word) if '.' in word else int(word)
    return True
  except ValueError:
    return False

In [22]:
def isInitial(word):
  return len(word) == 2 and word[1] == "."

In [25]:
class RakeKeywordExtractor:

  def __init__(self):
    self.stopwords = set(nltk.corpus.stopwords.words())
    self.top_fraction = 1 # consider top third candidate keywords by score

  def _generate_candidate_keywords(self, sentences):
    phrase_list = []
    for sentence in sentences:
      words = map(lambda x: "|" if x in self.stopwords or x in stop_list or isNumeric(x) or isInitial(x) else x,
        nltk.word_tokenize(sentence.lower()))
      phrase = []
      for word in words:
        if word == "|" or isPunct(word):
          if len(phrase) > 0:
            phrase_list.append(phrase)
            phrase = []
        else:
          phrase.append(word)
    return phrase_list
## СТОП!!
  def _calculate_word_scores(self, phrase_list):
    word_freq = nltk.FreqDist()
    word_degree = nltk.FreqDist()
    for phrase in phrase_list:
      #degree = len(filter(lambda x: not isNumeric(x), phrase)) - 1
      degree = len(list(filter(lambda x: not isNumeric(x), phrase))) - 1
      for word in phrase:
        word_freq[word] += 1 ## word_fd.inc(word.lower()) with word_fd[word.lower()] += 1
        word_degree[word] += degree # other words
    for word in word_freq.keys():
      word_degree[word] = word_degree[word] + word_freq[word] # itself
    # word score = deg(w) / freq(w)
    word_scores = {}
    for word in word_freq.keys():
      word_scores[word] = word_degree[word] / word_freq[word]
    return word_scores

  def _calculate_phrase_scores(self, phrase_list, word_scores):
    phrase_scores = {}
    for phrase in phrase_list:
      phrase_score = 0
      for word in phrase:
        phrase_score += word_scores[word]
      phrase_scores[" ".join(phrase)] = phrase_score
    return phrase_scores

  def extract(self, text, incl_scores=False):
    sentences = nltk.sent_tokenize(text)
    phrase_list = self._generate_candidate_keywords(sentences)
    word_scores = self._calculate_word_scores(phrase_list)
    phrase_scores = self._calculate_phrase_scores(
      phrase_list, word_scores)
    #sorted_phrase_scores = sorted(phrase_scores.iteritems(),
    sorted_phrase_scores = sorted(phrase_scores.items(),
      key=operator.itemgetter(1), reverse=True)
    n_phrases = len(sorted_phrase_scores)
    if incl_scores:
      return sorted_phrase_scores[0:int(n_phrases/self.top_fraction)]
    else:
      return map(lambda x: x[0],
        sorted_phrase_scores[0:int(n_phrases/self.top_fraction)])

In [26]:
rake = RakeKeywordExtractor()

In [32]:
with open('Text_MA_tokenized.txt', 'r', encoding='utf-8') as f:
    txt = f.read()

In [33]:
keywords = rake.extract(txt, incl_scores=True)

In [34]:
for eachkwrd in keywords:
    print(eachkwrd[0], eachkwrd[1])

202421собой вариант семантической компрессии 16.0
предложенным челнами организационных комитетов 16.0
благодаря использованию программных инструментов 15.0
способствующей формированию информационно-поискового портрета 15.0
терминологии рассматриваемой предметной области 14.5
согласно использованному издательскому шаблону 14.333333333333334
автоматическом выделении структурированных данных 14.0
широкий тематический диапазон текстов 13.741935483870968
₋автоматическое выделение ключевых слов 13.120000000000001
ткиклавтоматическое выделение ключевых слов 13.120000000000001
стандартный набор именованных сущностей 13.03888888888889
₋наибольшее среднее количество токенов 12.777777777777779
разметка уникальных именованных сущностей 12.503174603174603
автоматическая разметка именованных сущностей 12.003174603174603
следующие особые именованные сущности 11.742857142857144
автоматическая генерация ключевых слов 11.42
авторские наборы ключевых слов 11.370000000000001
качественная аннотация научной

In [35]:
keywords

[('202421собой вариант семантической компрессии', 16.0),
 ('предложенным челнами организационных комитетов', 16.0),
 ('благодаря использованию программных инструментов', 15.0),
 ('способствующей формированию информационно-поискового портрета', 15.0),
 ('терминологии рассматриваемой предметной области', 14.5),
 ('согласно использованному издательскому шаблону', 14.333333333333334),
 ('автоматическом выделении структурированных данных', 14.0),
 ('широкий тематический диапазон текстов', 13.741935483870968),
 ('₋автоматическое выделение ключевых слов', 13.120000000000001),
 ('ткиклавтоматическое выделение ключевых слов', 13.120000000000001),
 ('стандартный набор именованных сущностей', 13.03888888888889),
 ('₋наибольшее среднее количество токенов', 12.777777777777779),
 ('разметка уникальных именованных сущностей', 12.503174603174603),
 ('автоматическая разметка именованных сущностей', 12.003174603174603),
 ('следующие особые именованные сущности', 11.742857142857144),
 ('автоматическая ге

In [36]:
with open('Rake_2024_MitrAdam_Text_keywords.csv', 'w', encoding='utf-8') as f2:
    for eachkwrd in keywords:
        f2.write(str(eachkwrd[0])+";"+str(eachkwrd[1])+"\n")

KeyBert — это библиотека, которая использует модель BERT (Bidirectional Encoder Representations from Transformers):

Алгоритм использует модели на основе трансформеров для понимания контекста в предложениях.
Он оценивает схожесть между исходным текстом и кандидатными словами с помощью cosine similarity (косинусное сходство).
Библиотека выделяет те слова, которые наиболее контекстуально релевантны для темы текста. Это позволяет выделить не только часто встречающиеся термины, но и те, которые имеют сильное семантическое значение для текста.

In [ ]:
pip install keybert


In [37]:
#KeyBERT
import keybert

C:\Users\alena\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [38]:
from keybert import KeyBERT

In [49]:
with open('IMS2024\MitrofanovaAdamova_IMS_2024_rus.txt', 'r', encoding='utf-8') as f:
    doc = f.read()

In [ ]:
kw_model = KeyBERT()

In [50]:
keywords = kw_model.extract_keywords(doc)

In [51]:
keywords

[('алгоритмов', 0.4228),
 ('проводил', 0.4063),
 ('программирования', 0.4027),
 ('предложениям', 0.3982),
 ('понятиями', 0.393)]

In [52]:
keywords = kw_model.extract_keywords(doc, top_n=10)
keywords

[('алгоритмов', 0.4228),
 ('проводил', 0.4063),
 ('программирования', 0.4027),
 ('предложениям', 0.3982),
 ('понятиями', 0.393),
 ('поискового', 0.3926),
 ('построения', 0.392),
 ('процесс', 0.3908),
 ('персоналии', 0.385),
 ('предложения', 0.384)]

In [53]:
with open('KeyBert_2024_MitrAdam_Text_kw.csv', 'w', encoding='utf-8') as f2:
    for eachkwrd in keywords:
        f2.write(str(eachkwrd[0])+";"+str(eachkwrd[1])+"\n")

In [54]:
keywords = kw_model.extract_keywords(doc, keyphrase_ngram_range=(1, 2), top_n=10)
keywords

[('процессе подготовки', 0.4776),
 ('поискового сервиса', 0.4685),
 ('широко применяется', 0.4639),
 ('помощи алгоритмов', 0.4621),
 ('вопросов или', 0.454),
 ('исходные предложения', 0.454),
 ('информационно поискового', 0.4513),
 ('основного содержания', 0.4458),
 ('зрения технологической', 0.4453),
 ('при использовании', 0.4421)]

In [55]:
with open('KeyBERT_MItrAdam_Text_bigram.csv', 'w', encoding='utf-8') as f2:
    for eachkwrd in keywords:
        f2.write(str(eachkwrd[0])+";"+str(eachkwrd[1])+"\n")

In [56]:
keywords = kw_model.extract_keywords(doc, keyphrase_ngram_range=(1, 3), top_n=10)
keywords

[('многопараметрического поиска источников', 0.5134),
 ('проведено сравнение алгоритмов', 0.5057),
 ('широко применяется различных', 0.5051),
 ('процедур автоматического выделения', 0.5006),
 ('руководителя петербургской школы', 0.4958),
 ('при помощи алгоритмов', 0.4838),
 ('процедурой процессе подготовки', 0.4832),
 ('алгоритмов предполагая суммаризация', 0.4822),
 ('процедур тематического моделирования', 0.4813),
 ('помощи алгоритмов суммаризации', 0.481)]

In [57]:
with open('KeyBERT_MitrAdam_Text_trigram.csv', 'w', encoding='utf-8') as f2:
    for eachkwrd in keywords:
        f2.write(str(eachkwrd[0])+";"+str(eachkwrd[1])+"\n")

In [ ]:
#RuTermExtract

ru-term-extract — это библиотека для выделения ключевых слов в текстах на русском языке. Она использует TF-IDF (Term Frequency-Inverse Document Frequency) для оценки значимости слов в тексте. Алгоритм работает следующим образом:

TF (Term Frequency): Рассчитывает частоту появления слова в тексте.
IDF (Inverse Document Frequency): Оценка того, насколько редким является слово в большом корпусе текстов.
Вычисляется произведение TF и IDF, чтобы выделить наиболее значимые термины в контексте всего документа.

In [7]:
!pip install rutermextract

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for rutermextract: filename=rutermextract-0.3-py3-none-any.whl size=9792 sha256=ab1d08e2453a5f41e82e3f086329f4c268bc24ab80c723f8a9f04fbd9ec8947d
  Stored in directory: c:\users\alena\appdata\local\pip\cache\wheels\44\00\bc\33a2ee393657898aed2be21b2598c4e2a7ababd404a2f979e1
Successfully built rutermextract



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from rutermextract import TermExtractor

In [9]:
term_extractor = TermExtractor()

In [13]:
with open('IMS2024\MitrofanovaAdamova_IMS_2024_rus.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [14]:
for term in term_extractor(text):
    print(term.normalized, term.count)

ключевые выражения 19
корпус 18
ткикл 12
тексты 10
аннотации 9
именованные сущности 8
ключевые слова 6
таблица 6
применение 5
использование 5
компьютерная лингвистика 4
абстрактивная суммаризация 4
текст 4
суммаризация 4
разметка 4
лемматизация 4
экстрактивная суммаризация 3
тексты корпуса 3
машинное обучение 3
именованные сущность 3
структура 3
статья 3
стати 3
словосочетания 3
рис 3
решение 3
работа 3
проведение 3
предложения 3
наборы 3
генерация 3
аннотация 3
рассматриваемая предметная область 2
экспертная разметка 2
помощь алгоритмов 2
оригинальное текст 2
ограниченный объём 2
нетекстовые элементы 2
неразмеченные тексты 2
необходимость восстановления 2
научные стати 2
научная статья 2
лемматизированные тексты 2
ключевые выражение 2
исходное текст 2
данная статья 2
вычислительные онтология 2
английский язык 2
авторские наборы 2
surname_conference name_year 2
этапы 2
читатели 2
ход 2
форма 2
упрощение 2
токены 2
состав 2
создание 2
сборники 2
сбор 2
рисунки 2
разработка 2
процесс 2
п

In [15]:
with open('RuTermExtract_keywords.csv', 'w', encoding='utf-8') as f2:
    for term in term_extractor(text):
        f2.write(str(term.normalized)+";"+str(term.count)+"\n")

In [ ]:
#SpaCy

In [ ]:
!pip install spacy

In [ ]:
!pip install https://github.com/explosion/spacy-models/releases/download/ru_core_news_sm-3.1.0/ru_core_news_sm-3.1.0.tar.gz

In [66]:
import spacy
from collections import Counter
from nltk.corpus import stopwords

In [67]:
# Для SpaCy (дополнительно взяли стоп-слова из nltk, с ними результаты улучшились)
spacy_model = spacy.load('ru_core_news_sm')

In [68]:
spacy_model.max_length = 2400000

In [69]:
nltk_stopwords = stopwords.words('russian')

In [70]:
spacy_stopwords = spacy_model.Defaults.stop_words

In [71]:
def extract_keywords_with_spacy(text):
    keywords = []
    doc = spacy_model(text.read())
    for token in doc:
        # Тексты уже очищены от знаков препинания и лемматизированы, тем не менее, остались некоторые символы,
        # которые нужно удалить
        if token.text not in spacy_stopwords and token.text not in nltk_stopwords and \
                token.text not in "`'«»...—-":
            if token.pos_ in ('ADJ', 'NOUN', 'VERB'):
                keywords.append(token.text)

    freq_word = Counter(keywords)
    max_freq = Counter(keywords).most_common(1)[0][1]
    for w in freq_word:
        freq_word[w] = round(freq_word[w] / max_freq, 3)

    freq_20 = freq_word.most_common(20)

    with open('spacykeywords.csv', 'w', encoding='utf-8') as f2:
        for term in freq_20:
            f2.write(str(term[0])+";"+str(term[1])+"\n")

    print(freq_20)

In [ ]:
with open('IMS2024\MitrofanovaAdamova_IMS_2024_rus.txt', 'r', encoding='utf-8') as file:
    extract_keywords_with_spacy(file)

[('статей', 1.0), ('докладов', 1.0), ('лингвистика', 1.0), ('гг.', 1.0), ('разметки', 1.0), ('Статья', 0.5), ('посвящена', 0.5), ('проблемам', 0.5), ('разработки', 0.5), ('корпуса', 0.5), ('корпусной', 0.5), ('лингвистике', 0.5), ('создаваемого', 0.5), ('кафедре', 0.5), ('математической', 0.5), ('лингвистики', 0.5), ('СПбГУ', 0.5), ('Корпус', 0.5), ('создан', 0.5), ('руководством', 0.5)]


In [74]:
with open('Spacy_MitrAdam_Text_keywords.csv', 'w', encoding='utf-8') as f2:
    for eachkwrd in keywords:
        f2.write(str(eachkwrd[0])+";"+str(eachkwrd[1])+"\n")

Yake


In [ ]:
! pip install yake


In [63]:
with open('IMS2024\MitrofanovaAdamova_IMS_2024_rus.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [64]:
import yake


# Настройки YAKE
language = "ru"  # Русский язык
max_ngram_size = 3  # Максимальный размер ключевой фразы (например: до триграмм)
deduplication_threshold = 0.9  # Порог схожести фраз
numOfKeywords = 10  # Сколько ключевых фраз извлекать

# Инициализация модели
kw_extractor = yake.KeywordExtractor(
    lan=language,
    n=max_ngram_size,
    dedupLim=deduplication_threshold,
    top=numOfKeywords,
    features=None
)

# Извлечение ключевых слов
keywords = kw_extractor.extract_keywords(text)

# Вывод результата
for kw, score in keywords:
    print(f"{kw} (score: {score:.4f})")


ключевых выражений (score: 0.0030)
выделения ключевых выражений (score: 0.0062)
текстов (score: 0.0082)
ключевых (score: 0.0085)
IMS (score: 0.0090)
Виктора Павловича Захарова (score: 0.0101)
корпуса (score: 0.0118)
выражений (score: 0.0126)
IMS CompLing (score: 0.0129)
корпуса ТКиКЛ (score: 0.0133)


In [65]:
with open('Yake_MitrAdam_Text_keywords.csv', 'w', encoding='utf-8') as f2:
    for eachkwrd in keywords:
        f2.write(str(eachkwrd[0])+";"+str(eachkwrd[1])+"\n")

TextRank

In [106]:
with open('IMS2024\MitrofanovaAdamova_IMS_2024_rus.txt', 'r', encoding='utf-8') as f:
    my_text = f.read()

In [ ]:
! pip install networkx


In [101]:
import networkx as nx
from nltk.tokenize import word_tokenize
from itertools import combinations

In [102]:
# Загружаем модели
morph = pymorphy2.MorphAnalyzer()
nlp = spacy.load("ru_core_news_sm")

def lemmatize_text(text):
    # Очистка
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    tokens = word_tokenize(text, language='russian')
    lemmas = [morph.parse(token)[0].normal_form for token in tokens if token.isalpha()]
    
    return " ".join(lemmas)

def extract_phrases_from_lemmas(lemmatized_text):
    doc = nlp(lemmatized_text)

    tokens = [token for token in doc 
              if token.pos_ in {"NOUN", "ADJ"} 
              and not token.is_stop 
              and token.is_alpha]

    phrases = []
    current_phrase = []
    for token in tokens:
        current_phrase.append(token.text)
        if token.pos_ == "NOUN":
            phrases.append(" ".join(current_phrase))
            current_phrase = []

    return phrases

def textrank_phrases(phrases, top_n=10):
    vocab = list(set(phrases))
    graph = nx.Graph()
    graph.add_nodes_from(vocab)

    window_size = 2
    for i in range(len(phrases) - window_size + 1):
        window = phrases[i:i+window_size]
        for w1, w2 in combinations(window, 2):
            if graph.has_edge(w1, w2):
                graph[w1][w2]['weight'] += 1
            else:
                graph.add_edge(w1, w2, weight=1)

    ranks = nx.pagerank(graph)
    ranked = sorted(ranks.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_n]

In [107]:
# Пайплайн
lemmatized = lemmatize_text(my_text)
phrases = extract_phrases_from_lemmas(lemmatized)
keywords = textrank_phrases(phrases, top_n=10)

# Результат

for i, (phrase, score) in enumerate(keywords, 1):
    print(f"{i}. {phrase} (score: {score:.4f})")

1. текст (score: 0.0498)
2. корпус (score: 0.0333)
3. ключевой выражение (score: 0.0162)
4. аннотация (score: 0.0157)
5. год (score: 0.0108)
6. сущность (score: 0.0102)
7. конференция (score: 0.0093)
8. алгоритм (score: 0.0092)
9. статья (score: 0.0090)
10. исследование (score: 0.0082)


In [108]:
with open('TextRank_MitrAdam_Text_keywords.csv', 'w', encoding='utf-8') as f2:
    for eachkwrd in keywords:
        f2.write(str(eachkwrd[0])+";"+str(eachkwrd[1])+"\n")

Сравнительный дата-фрейм

In [ ]:
! pip install sentence-transformers


In [111]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

csv_folder = 'все csv keywords'  

# путь к файлу с авторскими ключевыми словами
author_keywords_path = 'IMS2024\MitrofanovaAdamova_IMS_2024_KW_rus.txt'

# Порог сходства
threshold = 0.7

# Загружаем модель
model = SentenceTransformer('all-MiniLM-L6-v2')

# Загружаем авторские ключевые слова
with open(author_keywords_path, 'r', encoding='utf-8') as f:
    gold_keywords = [kw.strip().lower() for kw in f.read().split(',') if kw.strip()]

results = []

#  Проходим по всем CSV-файлам в папке
for filename in os.listdir(csv_folder):
    if filename.endswith('.csv'):
        filepath = os.path.join(csv_folder, filename)
        method_name = filename.replace('.csv', '')

        # Загружаем CSV с учетом ; как разделителя
        df = pd.read_csv(filepath, sep=';', header=None, usecols=[0], names=['Keyword'], engine='python', on_bad_lines='skip')

        predicted_keywords = df['Keyword'].astype(str).str.strip().str.lower().tolist()

        # Эмбеддинги
        gold_emb = model.encode(gold_keywords, convert_to_tensor=True)
        pred_emb = model.encode(predicted_keywords, convert_to_tensor=True)

        # Сходство
        cos_sim = util.cos_sim(pred_emb, gold_emb)

        matches = 0
        matched_gold = set()

        for i in range(len(predicted_keywords)):
            for j in range(len(gold_keywords)):
                if cos_sim[i][j] >= threshold and j not in matched_gold:
                    matches += 1
                    matched_gold.add(j)
                    break  # учитывать предсказание один раз

        precision = matches / len(predicted_keywords) if predicted_keywords else 0
        recall = matches / len(gold_keywords) if gold_keywords else 0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        results.append({
            'Алгоритм': method_name,
            'Precision': round(precision, 3),
            'Recall': round(recall, 3),
            'F1': round(f1, 3)
        })

# Таблица результатов
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='F1', ascending=False).reset_index(drop=True)

print("Результаты сравнения алгоритмов:\n")
print(results_df.to_string(index=False))


Результаты сравнения алгоритмов:

                             Алгоритм  Precision  Recall    F1
      TextRank_MitrAdam_Text_keywords      0.500   0.714 0.588
     KeyBERT_MItrAdam_Abstract_bigram      0.400   0.571 0.471
    KeyBERT_MitrAdam_Abstract_trigram      0.400   0.571 0.471
         KeyBert_MitrAdam_Abstract_kw      0.400   0.571 0.471
  TextRank_MitrAdam_Abstract_keywords      0.400   0.571 0.471
      Yake_MitrAdam_Abstract_keywords      0.400   0.571 0.471
         KeyBERT_MItrAdam_Text_bigram      0.400   0.571 0.471
        KeyBERT_MitrAdam_Text_trigram      0.300   0.429 0.353
Ruterm_Abstract_Mitr_Adamova_keywords      0.200   1.000 0.333
     Spacy_MitrAdam_Abstract_keywords      0.200   0.286 0.235
          Yake_MitrAdam_Text_keywords      0.200   0.286 0.235
         Spacy_MitrAdam_Text_keywords      0.200   0.286 0.235
      Rake_MitrAdam_Abstract_keywords      0.130   1.000 0.230
             KeyBert_MitrAdam_Text_kw      0.100   0.143 0.118
    Ruterm_Text_Mitr-